# Day 086 Solution — Research Agent over AI Documents

In [ ]:
import json
from dataclasses import dataclass, field
@dataclass
class Document:
    content: str
    metadata: dict = field(default_factory=dict)

class SimpleRetriever:
    def __init__(self):
        self._docs = []
    def add(self, doc):
        self._docs.append(doc); return self
    def add_all(self, docs):
        for d in docs: self._docs.append(d)
        return self
    def _score(self, query, doc):
        q = set(query.lower().split())
        d = set(doc.content.lower().split())
        return len(q & d) / (len(q | d) + 1e-9)
    def search(self, query, top_k=3):
        if not self._docs: return []
        return sorted(self._docs, key=lambda doc: self._score(query, doc), reverse=True)[:top_k]
    def __len__(self): return len(self._docs)
def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    start = str(text).find("{")
    end   = str(text).rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except (json.JSONDecodeError, ValueError):
        return None
def format_docs(docs):
    if not docs:
        return "No documents found."
    lines = []
    for i, d in enumerate(docs, 1):
        source = d.metadata.get("source", f"doc{i}")
        lines.append(f"[{i}] ({source}) {d.content}")
    return "\n".join(lines)

def build_retrieval_prompt(question, docs):
    context = format_docs(docs)
    system = "\n".join([
        "You are a helpful assistant.",
        "Answer the question using ONLY the provided documents.",
        "If the answer is not in the documents, say: I don't have enough information.",
        "Cite document numbers like [1] when referencing specific facts.",
    ])
    user = "Documents:\n" + context + "\n\nQuestion: " + str(question)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def retrieve_and_answer(question, retriever, top_k=3, llm_fn=None):
    docs = retriever.search(question, top_k=top_k)
    prompt = build_retrieval_prompt(question, docs)
    answer = call_llm(prompt, llm_fn=llm_fn)
    return {"question": question, "docs": docs, "answer": answer}
def build_agent_step_prompt(question, context):
    has_context = bool(context) and context != "No documents found."
    ctx_line = "Current context:\n" + context if has_context else "No context retrieved yet."
    system = "\n".join([
        "You are a research agent. Decide your next action.",
        'To search: {"action": "retrieve", "query": "your search query"}',
        'To answer: {"action": "answer",   "text":  "your final answer"}',
        "Use retrieve to gather information; use answer when you have enough.",
        "Reply with ONLY valid JSON.",
    ])
    user = "Question: " + str(question) + "\n\n" + ctx_line
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def parse_agent_action(text):
    data = safe_parse_json(text) or {}
    if data.get("action") == "answer":
        return {"action": "answer", "text": str(data.get("text", ""))}
    query = str(data.get("query", ""))
    return {"action": "retrieve", "query": query}
def _mock_retrieval_llm(script):
    idx = [0]
    def _fn(messages):
        i = min(idx[0], len(script) - 1)
        idx[0] += 1
        return json.dumps(script[i])
    return _fn
class RetrievalAgent:
    def __init__(self, retriever, llm_fn=None, top_k=3, max_iterations=5):
        self.retriever = retriever
        self._llm_fn = llm_fn
        self._top_k = top_k
        self.max_iterations = max_iterations
        self._history = []
    def ask(self, question):
        all_docs = []; steps = []
        for iteration in range(self.max_iterations):
            context = format_docs(all_docs) if all_docs else "No documents retrieved yet."
            prompt = build_agent_step_prompt(question, context)
            response = call_llm(prompt, llm_fn=self._llm_fn)
            act = parse_agent_action(response)
            if act["action"] == "answer":
                answer = act.get("text", "")
                steps.append({"step": iteration + 1, "action": "answer", "text": answer})
                record = {"question": question, "docs": all_docs, "answer": answer, "steps": steps}
                self._history.append(record); return record
            query = act.get("query", question)
            docs = self.retriever.search(query, top_k=self._top_k)
            all_docs.extend(docs)
            steps.append({"step": iteration + 1, "action": "retrieve", "query": query, "found": len(docs)})
        ans_prompt = build_retrieval_prompt(question, all_docs)
        answer = call_llm(ans_prompt, llm_fn=self._llm_fn)
        steps.append({"step": self.max_iterations + 1, "action": "answer", "text": answer})
        record = {"question": question, "docs": all_docs, "answer": answer, "steps": steps}
        self._history.append(record); return record
    def history(self): return list(self._history)
    def clear_history(self): self._history.clear()


In [ ]:
retriever = SimpleRetriever()
docs = [
    Document('Neural networks are modelled loosely on the human brain with layers of interconnected nodes.', {"source": 'ai_basics'}),
    Document('Deep learning is a subset of machine learning that uses neural networks with many layers.', {"source": 'deep_learning'}),
    Document('Transformers use self-attention mechanisms to process sequences in parallel.', {"source": 'transformers'}),
    Document('BERT is a transformer model pre-trained on masked language modelling and next sentence prediction.', {"source": 'bert'}),
    Document('GPT models are decoder-only transformers trained to predict the next token.', {"source": 'gpt'}),
    Document('Reinforcement learning trains agents by rewarding desired behaviour and penalising undesired behaviour.', {"source": 'rl'}),
    Document('RAG (Retrieval-Augmented Generation) combines search with language model generation.', {"source": 'rag'}),
    Document('Embeddings map words or sentences to dense numerical vectors that capture meaning.', {"source": 'embeddings'}),
    Document('Fine-tuning adapts a pre-trained model to a specific task using a smaller labelled dataset.', {"source": 'fine_tuning'}),
    Document('Prompt engineering is the practice of crafting inputs to elicit better outputs from language models.', {"source": 'prompting'}),
]
retriever.add_all(docs)
print(f'Loaded {len(retriever)} documents')


In [ ]:
def _mock_retrieval_llm(script):
    idx = [0]
    def _fn(messages):
        i = min(idx[0], len(script) - 1)
        idx[0] += 1
        return json.dumps(script[i])
    return _fn

# script-driven LLM for gate testing
_scripts = [[{"action": "retrieve", "query": "transformer attention mechanism"}, {"action": "answer", "text": "Transformers use self-attention mechanisms to process sequences in parallel [3]."}], [{"action": "retrieve", "query": "GPT model architecture"}, {"action": "answer", "text": "GPT models are decoder-only transformers trained to predict the next token [5]."}], [{"action": "retrieve", "query": "RAG retrieval generation"}, {"action": "answer", "text": "RAG combines search with language model generation [7]."}]]
_script_states = [[0] for _ in _scripts]
_qidx = [0]

def _solution_llm(messages):
    si = _qidx[0] % len(_scripts)
    state = _script_states[si]
    script = _scripts[si]
    i = min(state[0], len(script) - 1)
    state[0] += 1
    return json.dumps(script[i])


In [ ]:
questions = ["How do transformers work?", "What is a GPT model?", "What is RAG?"]

for q in questions:
    _qidx[0] = questions.index(q)
    for state in _script_states: state[0] = 0
    agent = RetrievalAgent(retriever, llm_fn=_solution_llm, max_iterations=5)
    r = agent.ask(q)
    print(f"Q: {r['question']}")
    print(f"  Steps: {len(r['steps'])}")
    print(f"  Answer: {r['answer'][:100]}")
    print()


In [ ]:
# Smoke-test: single agent with full scripted run
_for_test = RetrievalAgent(
    retriever,
    llm_fn=_mock_retrieval_llm([
        {"action": "retrieve", "query": "deep learning neural networks"},
        {"action": "answer",   "text":  "Deep learning uses neural networks with many layers [2]."},
    ]),
    max_iterations=5,
)
_r = _for_test.ask("What is deep learning?")
assert len(_r["steps"]) == 2
assert "Deep learning" in _r["answer"]
assert len(_for_test.history()) == 1
_for_test.clear_history()
assert _for_test.history() == []
print("Solution smoke-test passed.")
